# Helper Functions

In [2]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns


def get_metrics_df(data):
    filtered_data = {k: {metric: v for metric, v in metrics.items() 
                     if metric not in ['question', 'ground_truth', 'output']} 
                 for k, metrics in data.items()}
    df = pd.DataFrame.from_dict(filtered_data, orient='index')

    return df


def check_normality(metrics):
    # Run Shapiro-Wilk test for normality
    normality_results = {}
    for column in metrics.columns:
        stat, p_value = stats.shapiro(metrics[column])
        normality_results[column] = {
            'Shapiro-Wilk Statistic': stat, 
            'p-value': p_value,
        }

    normality_df = pd.DataFrame(normality_results).T
    print("Shapiro-Wilk Test Results:")
    print(normality_df)

    # Visualize normality with histograms and Q-Q plots
    for column in metrics.columns:
        plt.figure(figsize=(10, 4))

        # Histogram with KDE
        plt.subplot(1, 2, 1)
        sns.histplot(metrics[column], kde=True)
        plt.title(f'Histogram of {column}')

        # Q-Q plot
        plt.subplot(1, 2, 2)
        stats.probplot(metrics[column], dist="norm", plot=plt)
        plt.title(f'Q-Q Plot of {column}')

        plt.tight_layout()
        plt.show()
    
    return normality_df

def t_test(metrics1, metrics2):
    t_test_results = {}
    mean_diffs = metrics1.mean() - metrics2.mean()
    
    for column in metrics2.columns:
        t_stat, p_value = stats.ttest_rel(metrics1[column], metrics2[column])
        t_test_results[column] = {
            't-statistic': t_stat,
            'p-value': p_value,
            'significant': p_value <= 0.05,
            'better_results': 'first metric' if mean_diffs[column] > 0 else 'second metric',
            'mean_difference': mean_diffs[column],
        }
    
    t_test_df = pd.DataFrame(t_test_results).T
    t_test_df["significant"] = t_test_df["significant"].astype(bool)
    t_test_df["better_results"] = t_test_df["better_results"].astype(str)
    
    sig_count = t_test_df['significant'].sum()
    first_count = (t_test_df['better_results'] == 'first metric').sum()
    second_count = (t_test_df['better_results'] == 'second metric').sum()
    
    print(f"Significant on {sig_count} out of {len(t_test_df)} --> {sig_count / len(t_test_df) * 100:.2f}%")
    print(f"First metric preferred in {first_count} metrics ; second in {second_count} metrics")
    return t_test_df

def wilcoxon_test(metrics1, metrics2):
    wilcoxon_results = {}
    for column in metrics2.columns:
        stat, p_value = stats.wilcoxon(metrics2[column], metrics1[column])
        wilcoxon_results[column] = {'Wilcoxon Statistic': stat, 'p-value': p_value}
    wilcoxon_df = pd.DataFrame(wilcoxon_results).T
    wilcoxon_df["significant"] = wilcoxon_df["p-value"] <= 0.05
    return wilcoxon_df


# Analysis

In [4]:
from utils.utils import load_from_pickle

directories = [
    "baseline",
    "cos_512_128_bottom", "cos_512_128_alternate", "cos_512_128_top", "cos_512_128_nocontext",
    "cos_256_64_top", "cos_1024_256_top", "sci_512_128_nonoise", "sci_512_128_single",
    "sci_512_128_multiple", "sci_256_64_nonoise", "sci_1024_256_nonoise", "sp_zero",
    "sp_few", "sp_cot", "llmtemp_low", "llmtemp_mid", "llmtemp_high", "llmtopp_low",
    "llmtopp_mid", "llmtopp_high", "llmmodel_llama3b",
    "sptp_zll", "sptp_fll", "sptp_cll", "sptp_zml", "sptp_zhl", "sptp_zlm", "sptp_zlh",
    "sptp_zmm", "sptp_zmh", "sptp_zhm", "sptp_zhh", "sptp_fml", "sptp_fhl", "sptp_flm",
    "sptp_fmm", "sptp_fmh", "sptp_fhh", "sptp_cml", "sptp_chl", "sptp_flh", "sptp_fhm",
    "sptp_clm", "sptp_clh", "sptp_cmm", "sptp_cmh", "sptp_chm", "sptp_chh",
    "cs_nc", "cs_nn", "cs_bs", "cs_bm", "cs_as", "cs_am", "cs_ts", "cs_tm"
]

metrics = {}

for dir_name in directories:
    path = f"llm_experiments/{dir_name}"
    data = load_from_pickle(path)
    metrics[dir_name] = get_metrics_df(data)

In [5]:
metrics['sptp_zll'].describe()

,BLEU,ROUGE-1_Precision,ROUGE-1_Recall,ROUGE-1_F1,ROUGE-2_Precision,ROUGE-2_Recall,ROUGE-2_F1,ROUGE-L_Precision,ROUGE-L_Recall,ROUGE-L_F1,METEOR,BERTScore_Precision,BERTScore_Recall,BERTScore_F1,RAGAS_Answer_Similarity
count,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000,2644.000000
mean,0.070597,0.151683,0.343822,0.196182,0.099396,0.235515,0.130822,0.135997,0.310203,0.176302,0.207115,0.773702,0.860923,0.814577,0.781641
std,0.104816,0.142296,0.324811,0.177934,0.120278,0.281948,0.153148,0.126993,0.294938,0.159561,0.219867,0.050123,0.045110,0.044643,0.106264
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.587639,0.724004,0.678361,0.255460
25%,0.003467,0.043478,0.086957,0.057406,0.000000,0.000000,0.000000,0.041885,0.083333,0.054054,0.032547,0.731268,0.828146,0.777314,0.734000
50%,0.021735,0.111111,0.221576,0.141011,0.056604,0.117995,0.074350,0.100610,0.200000,0.129491,0.123455,0.772333,0.850264,0.810616,0.797276
75%,0.099438,0.223159,0.572321,0.301499,0.146905,0.362563,0.202455,0.196050,0.476190,0.265442,0.334657,0.808585,0.890369,0.846251,0.852406
max,0.804224,0.833333,1.000000,0.909091,0.826087,1.000000,0.904762,0.833333,1.000000,0.909091,0.978177,0.940794,0.987243,0.963459,0.980691


In [79]:
# normality_df = check_normality(metrics)

In [6]:
t_test_df = t_test(metrics['cs_as'], metrics['cs_ts'])
print(t_test_df)

Significant on 9 out of 15 --> 60.00%
First metric preferred in 0 metrics ; second in 15 metrics
                        t-statistic   p-value  significant better_results  \
BLEU                      -0.813472  0.416021        False  second metric   
ROUGE-1_Precision          -3.14865  0.001659         True  second metric   
ROUGE-1_Recall             -1.86825  0.061838        False  second metric   
ROUGE-1_F1                -2.850072  0.004405         True  second metric   
ROUGE-2_Precision         -2.961216  0.003092         True  second metric   
ROUGE-2_Recall            -2.170886  0.030029         True  second metric   
ROUGE-2_F1                 -2.71352    0.0067         True  second metric   
ROUGE-L_Precision         -2.805769  0.005056         True  second metric   
ROUGE-L_Recall            -1.822904  0.068431        False  second metric   
ROUGE-L_F1                -2.553339  0.010725         True  second metric   
METEOR                    -1.457001  0.145235        Fal

In [35]:

wilcoxon_df = wilcoxon_test(metrics, metrics2)
print(wilcoxon_df)

                         Wilcoxon Statistic   p-value  significant
BLEU                              1198509.5  0.400090        False
ROUGE-1_Precision                 1223181.0  0.371218        False
ROUGE-1_Recall                     703676.5  0.562967        False
ROUGE-1_F1                        1231974.0  0.520608        False
ROUGE-2_Precision                 1170497.0  0.774836        False
ROUGE-2_Recall                     594490.0  0.914557        False
ROUGE-2_F1                        1173822.5  0.863271        False
ROUGE-L_Precision                 1234785.5  0.486980        False
ROUGE-L_Recall                     678512.0  0.669509        False
ROUGE-L_F1                        1244461.5  0.600046        False
METEOR                            1220363.0  0.631338        False
BERTScore_Precision               1574257.5  0.231189        False
BERTScore_Recall                  1612335.0  0.496395        False
BERTScore_F1                      1574480.5  0.306412        F